In [71]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re

%config InlineBackend.figure_format = 'svg'

def read_adf07_ionization_file(filepath, species_key = 'N'):
    """
    Parses ADAS ADF07 ionization coefficient files (e.g., szd96#n_n0.dat).
    These files contain electron impact ionization rate coefficients.
    """
    try:
        with open(filepath, 'r') as f:
            content = f.read()
    except FileNotFoundError:
        # This is expected if a file for a certain charge state doesn't exist.
        return None
    
    # Split into lines
    lines = content.splitlines()
    
    # Find data blocks (they start with charge state info like "N + 0/N + 1/")
    data_blocks = []
    i = 0
    while i < len(lines):
        line = lines[i]
        # Look for lines with charge state transitions for Nitrogen
        if f'{species_key} +' in line and '/' in line and 'I.P.' in line:
            # Parse header
            header = line
            charge_info = header.split('/')[0:2]
            initial_state = charge_info[0].strip()
            final_state = charge_info[1].strip()
            
            # Extract ionization potential if present
            ip_match = re.search(r'I\.P\.\s*=\s*([\d.]+)', header)
            ip_cm = float(ip_match.group(1)) if ip_match else None
            
            # Next lines contain temperature grid
            i += 1
            temp_values = []
            # Find all temperature and rate values, which are listed consecutively
            all_values_str = ""
            while i < len(lines) and not (lines[i].startswith('C--') or lines[i].strip() == '' or 'ISEL' in lines[i]):
                # Replace Fortran's 'D' with 'E' for scientific notation
                all_values_str += lines[i].replace('D', 'E') + " "
                i += 1
            
            all_values = [float(x) for x in all_values_str.split()]
            num_points = len(all_values) // 2

            if num_points > 0:
                temp_values = all_values[:num_points]
                rate_values = all_values[num_points:]

                if len(temp_values) == len(rate_values):
                    data_blocks.append({
                        'initial': initial_state,
                        'final': final_state,
                        'ip_cm': ip_cm,
                        'temp_eV': np.array(temp_values),
                        'rate_cm3_s': np.array(rate_values)
                    })
        else:
            i += 1
    
    return data_blocks

def parse_all_adf07_files(download_dir, species_key = 'N'):
    """
    Parse all ADF07 nitrogen ionization files (szd96#n_n0.dat through szd96#n_n6.dat).
    """
    all_data = {}
    
    for charge_state in range(7):  # N0 through N6
        filename = f"{species_key}{charge_state}.dat"
        filepath = download_dir / filename
        
        if not filepath.exists():
            continue

        print(f"Reading {filename}...")
        data_blocks = read_adf07_ionization_file(filepath, species_key)
        
        if data_blocks:
            # Use the first data block (total ionization rate, usually ISEL=1)
            block = data_blocks[0]
            
            # Create DataFrame
            df = pd.DataFrame({
                'Te_eV': block['temp_eV'],
                'Rate_cm3_s': block['rate_cm3_s']
            })
            
            # Store with transition key
            state_key = f'{species_key}{charge_state}+ -> {species_key}{charge_state+1}+'
            all_data[state_key] = df
            
            print(f"  Found data for {state_key}: {len(df)} temperature points")

    return all_data

def calculate_cpp_rate(Te_eV, E_ioniz_eV, a_ioniz, q_ioniz):
    """Calculates the ionization rate using a modified Arrhenius formula."""
    Te_eV_safe = np.maximum(Te_eV, 1e-99)
    rate = a_ioniz * (Te_eV_safe ** q_ioniz) * np.exp(-E_ioniz_eV / Te_eV_safe)
    return rate

def fit_ionization_coefficient(adas_df, E_ioniz_eV, fit_temp_eV, q_ioniz):
    """
    Fits the a_ioniz coefficient at a specific temperature for a given q.
    """
    # Interpolate ADAS rate at the fitting temperature
    adas_rate_at_fit_temp = np.interp(fit_temp_eV, adas_df['Te_eV'], adas_df['Rate_cm3_s'])
    
    # Calculate the a_ioniz coefficient
    denominator = (fit_temp_eV ** q_ioniz) * np.exp(-E_ioniz_eV / fit_temp_eV)
    if denominator > 0:
        a_ioniz_fit = adas_rate_at_fit_temp / denominator
    else:
        a_ioniz_fit = 0.0
    
    return a_ioniz_fit, adas_rate_at_fit_temp

In [72]:
home_dir = Path.home()
download_dir = Path("/data/dust/user/benahmed/src/hydro_castro/castro_sim/theory/impact_ionization/")

print("="*60)
print("ADAS ADF07 Nitrogen Ionization Data Parser")
print(f" (Searching for files in: {download_dir})")
print("="*60)
print()

# Parse all available ADF07 files from the downloads directory
adas_data = parse_all_adf07_files(download_dir, species_key='H')

if adas_data:
    print("\n" + "="*60)
    print("Fitting ionization coefficients to C++ formula")
    print("="*60)
    
    # Define ionization energies and fitting parameters for Nitrogen
    # You may need to adjust q_ioniz and fit_temp_eV for best results
    fit_params = {
        'H0+ -> H1+':  {'E_ioniz_eV': 13.60,  'q_ioniz': 0.4, 'fit_temp_eV': 10.0},
    }
    
    new_cpp_coeffs = {}
    available_states = [key for key in fit_params.keys() if key in adas_data]
    num_states = len(available_states)
    
    if num_states > 0:
        # Setup figure
        ncols = 2 if num_states > 1 else 1
        nrows = (num_states + 1) // 2 if num_states > 1 else 1
        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(8, 6), squeeze=False)
        fig.suptitle("Hydrogen Ionization Rates: ADAS Data vs C++ Formula Fit", fontsize=16, fontweight='bold')
        axes = axes.flatten()
        
        for idx, state_key in enumerate(available_states):
            df = adas_data[state_key]
            params = fit_params[state_key]
            ax = axes[idx]
            
            E_ioniz, q_ioniz, fit_temp = params['E_ioniz_eV'], params['q_ioniz'], params['fit_temp_eV']
            
            a_ioniz_fit, adas_rate_at_fit = fit_ionization_coefficient(df, E_ioniz, fit_temp, q_ioniz)
            new_cpp_coeffs[state_key] = (a_ioniz_fit, q_ioniz, E_ioniz)
            
            # Plotting
            ax.loglog(df['Te_eV'], df['Rate_cm3_s'], 'ko-', label='ADAS Data', markersize=4, lw=2)
            te_fine = np.logspace(np.log10(df['Te_eV'].min()), np.log10(df['Te_eV'].max()), 200)
            cpp_rates = calculate_cpp_rate(te_fine, E_ioniz, a_ioniz_fit, q_ioniz)
            ax.loglog(te_fine, cpp_rates, 'r--', label=f'Fit: A={a_ioniz_fit:.2e},  e={E_ioniz},  q={q_ioniz}', lw=2)
            ax.plot(fit_temp, adas_rate_at_fit, 'gs', markersize=8, label=f'Fit point ({fit_temp:.0f} eV)')
            
            print(f"\n{state_key}: (E_ioniz = {E_ioniz:.2f} eV)")
            print(f"  Fit at T = {fit_temp:.1f} eV with q = {q_ioniz:.3f}")
            print(f"  Fitted a_ioniz = {a_ioniz_fit:.3e}")
            
            # Formatting
            ax.set_title(state_key, fontweight='bold')
            ax.set_xlabel("Electron Temperature (eV)")
            ax.set_ylabel("Rate Coefficient (cm³/s)")
            ax.grid(True, which='both', ls=':', alpha=0.3)


        # Hide unused subplots
        for i in range(num_states, len(axes)):
            axes[i].set_visible(False)

        def model(Te, a, e, q):
            return a * (Te ** q) * np.exp(-e / Te)
        Te = np.logspace(0, 3, 1000)
        ax.plot(Te, model(Te, 7.1e-15 * 1e6, 13.6, 0.4), 'b-', label='From Mathis paper')
        ax.plot(Te, model(Te, 9.02e-15 * 1e6, 13.6, 0.332), 'b-', label='Cross ')
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        ax.legend(loc='best', fontsize=9)
        plt.savefig('hydrogen_ionization_fits.png', dpi=400)
        plt.show()

    # Output final coefficients
    print("\n" + "="*60)
    print("RECOMMENDED C++ COEFFICIENTS FOR problem_source.H")
    print("="*60)
    print("\n// Hydrogen collisional ionization rate coefficients")
    print("// Fitted from ADAS ADF07 data files (szd96#n_n*.dat)")
    print("// Rate = a_ioniz * (Te_eV^q_ioniz) * exp(-E_ioniz_eV/Te_eV)")

In [160]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sc
a, b, c, P = 4.0, 0.6, 0.56, 13.6

def rate_coefficient(T, a, b, c, P):
    """
    Compute the rate coefficient S (in cm^3 s^-1).
    """
    a = np.asarray(a)
    b = np.asarray(b)
    c = np.asarray(c)
    P = np.asarray(P)

    def exp_func(x):
        return np.exp(-x) / x
    
    int_1 = sc.integrate.quad(exp_func, P / T, np.inf)[0]
    int_2 = sc.integrate.quad(exp_func, P / T + c, np.inf)[0]
    
    S = 6.7e-7 * a / T**1.5 * 1/(P / T) * (int_1 - b * np.exp(c) / (P/T + c) * int_2)

    return S

Te_plot = np.logspace(0, 4, 1000)
S_plot = [rate_coefficient(T, a, b, c, P) for T in Te_plot]

plt.figure(figsize=(10, 6))
plt.loglog(Te_plot, S_plot, 'r-', label='Lotz Formula (Semi-analytical)', lw=2)
plt.loglog(df['Te_eV'], df['Rate_cm3_s'], 'ko-', label='ADAS (Exp Bell et al.)', markersize=4, lw=2)

def model(Te, a, e, q):
    return a * (Te ** q) * np.exp(-e / Te)
Te = np.logspace(0, 4, 1000)
plt.plot(Te, model(Te, 7.1e-15 * 1e6, 13.6, 0.4), 'g-', label='DI (Broks et al. )')
plt.plot(Te, model(Te, 12.4e-15 * 1e6, 13.6, 0.3), 'm-', label='EAU (Broks et al. ) ')
plt.plot(Te, model(Te, 7.1e-15 * 1e6, 13.6, 0.4) + model(Te, 12.4e-15 * 1e6, 13.6, 0.3), 'b--', label='Effective (Broks et al. )')
plt.xlim(1, 1e4)
plt.grid(True, which='both', ls=':', alpha=0.3)
plt.xlabel("Electron Temperature (eV)", fontsize = 14)
plt.ylabel("Rate Coefficient (cm³/s)", fontsize = 14)
plt.legend(fontsize=15, loc= 'lower right')
plt.title('Hydrogen electron impact ionization Rate Coefficients', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()



a_ar = [4, 4, 4, 4]
b_ar = [0.6, 0.55, 0.5, 0.45]
c_ar = [0.4, 0.45, 0.5, 0.6]
P_ar = [15.76, 27.63, 40.74, 59.81]
spec = ['Lotz 0->1+', ' Lotz 1->2+', 'Lotz 2->3+', 'Lotz 3->4+']
Te_plot = np.logspace(0, 4, 1000)

colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k']
plt.figure(figsize=(13, 8))
for i, coeff in enumerate(zip(a_ar, b_ar, c_ar, P_ar)):
    a, b, c, P = coeff
    S_plot = [rate_coefficient(T, a, b, c, P) for T in Te_plot]
    plt.loglog(Te_plot, S_plot, label=f'{spec[i]}', lw=2, color = colors[i])
    plt.fill_between(Te_plot, np.array(S_plot)*1.4, np.array(S_plot)*0.7, color=colors[i], alpha=0.2)

plt.grid(True, which='both', ls=':', alpha=0.3)
plt.ylim(1e-15, 1e-6)
# Temperatures (shared by all transitions)
temperatures = np.array([
    8.626E-01, 1.171E+00, 2.523E+00, 4.101E+00, 5.964E+00, 8.193E+00,
    1.090E+01, 1.426E+01, 1.852E+01, 2.406E+01, 3.152E+01, 4.202E+01,
    5.763E+01, 8.267E+01, 1.274E+02, 2.207E+02, 4.728E+02, 1.570E+03,
    1.611E+04, 8.652E+04
])

# Rates: one row per transition
rates = np.array([
    # AR+0 → AR+1
    [1.340E-16, 2.079E-14, 5.285E-11, 8.648E-10, 3.790E-09, 9.585E-09,
     1.830E-08, 2.964E-08, 4.323E-08, 5.876E-08, 7.579E-08, 9.381E-08,
     1.124E-07, 1.309E-07, 1.477E-07, 1.582E-07, 1.553E-07, 1.290E-07,
     6.628E-08, 3.646E-08],
    # AR+1 → AR+2
    [6.988E-21, 3.882E-14, 7.766E-11, 1.048E-09, 3.925E-09, 8.745E-09,
     1.508E-08, 2.241E-08, 3.004E-08, 3.766E-08, 4.530E-08, 5.285E-08,
     5.985E-08, 6.540E-08, 6.820E-08, 6.745E-08, 6.128E-08, 4.554E-08,
     1.944E-08, 1.215E-08],
    # AR+2 → AR+3
    [8.995E-22, 2.141E-14, 4.304E-11, 5.615E-10, 2.067E-09, 4.580E-09,
     7.852E-09, 1.160E-08, 1.559E-08, 1.959E-08, 2.340E-08, 2.680E-08,
     2.952E-08, 3.129E-08, 3.170E-08, 3.017E-08, 2.588E-08, 1.786E-08,
     7.086E-09, 5.189E-09],
    # AR+3 → AR+4
    [2.846E-21, 5.923E-15, 1.173E-11, 1.649E-10, 6.538E-10, 1.542E-09,
     2.784E-09, 4.304E-09, 6.024E-09, 7.862E-09, 9.731E-09, 1.152E-08,
     1.311E-08, 1.432E-08, 1.493E-08, 1.461E-08, 1.295E-08, 9.352E-09,
     3.978E-09, 3.467E-09]
])

# Plot all transitions
if False:
    for idx in range(rates.shape[0]):
        plt.loglog(temperatures, rates[idx] * 1e-1, label=f'ADAS Ar+{idx} → Ar+{idx+1}', linestyle='--', marker='o', color = colors[idx])

# Electron temperatures in eV for the first ISEL level (Ar+0 -> Ar+1)
Te_0 = np.array([0.8626, 1.171, 2.523, 4.101, 5.964, 8.193, 10.90, 14.26, 18.52, 24.06, 31.52, 42.02, 57.63, 82.67, 127.4, 220.7, 472.8, 1570., 16110., 86520.])
# Corresponding ionization rate coefficients in m^3/s
rate_0 = np.array([1.340e-16, 2.079e-14, 5.285e-11, 8.648e-10, 3.790e-09, 9.585e-09, 1.830e-08, 2.964e-08, 4.323e-08, 5.876e-08, 7.579e-08, 9.381e-08, 1.124e-07, 1.309e-07, 1.477e-07, 1.582e-07, 1.553e-07, 1.290e-07, 6.628e-08, 3.646e-08])

Te_1 = np.array([9.660E-01, 2.054E+00, 4.424E+00, 7.190E+00, 1.045E+01, 1.437E+01, 1.911E+01, 2.500E+01, 3.246E+01, 4.217E+01, 5.526E+01, 7.367E+01, 1.011E+02, 1.449E+02, 2.233E+02, 3.868E+02, 8.289E+02, 2.752E+03, 2.824E+04, 8.660E+04])
rate_1 = np.array([6.988E-21, 3.882E-14, 7.766E-11, 1.048E-09, 3.925E-09, 8.745E-09,1.508E-08, 2.241E-08, 3.004E-08, 3.766E-08, 4.530E-08, 5.285E-08,5.985E-08, 6.540E-08, 6.820E-08, 6.745E-08, 6.128E-08, 4.554E-08,1.944E-08, 1.215E-08])

Te_2 = np.array([1.365E+00, 3.027E+00, 6.523E+00, 1.060E+01, 1.542E+01, 2.118E+01,
 2.819E+01, 3.686E+01, 4.786E+01, 6.219E+01, 8.149E+01, 1.087E+02,
 1.490E+02, 2.137E+02, 3.292E+02, 5.704E+02, 1.222E+03, 4.058E+03,
 4.164E+04, 8.626E+04])

rate_2 = np.array([8.995E-22, 2.141E-14, 4.304E-11, 5.615E-10, 2.067E-09, 4.580E-09,
 7.852E-09, 1.160E-08, 1.559E-08, 1.959E-08, 2.340E-08, 2.680E-08,
 2.952E-08, 3.129E-08, 3.170E-08, 3.017E-08, 2.588E-08, 1.786E-08,
 7.086E-09, 5.189E-09])


Te_3 = np.array([2.163E+00, 4.445E+00, 9.574E+00, 1.556E+01, 2.264E+01, 3.109E+01,
 4.138E+01, 5.412E+01, 7.027E+01, 9.126E+01, 1.196E+02, 1.595E+02,
 2.187E+02, 3.138E+02, 4.833E+02, 8.373E+02, 1.794E+03, 5.957E+03,
 6.113E+04, 8.609E+04
])

rate_3 = np.array([2.846E-21, 5.923E-15, 1.173E-11, 1.649E-10, 6.538E-10, 1.542E-09,
 2.784E-09, 4.304E-09, 6.024E-09, 7.862E-09, 9.731E-09, 1.152E-08,
 1.311E-08, 1.432E-08, 1.493E-08, 1.461E-08, 1.295E-08, 9.352E-09,
 3.978E-09, 3.467E-09
])
plt.loglog(Te_0, rate_0 *1e-1, '-o', label='ADAS 0-> 1+ (Dere et al.)', color = 'blue')
plt.loglog(Te_1, rate_1 *1e-1, '-o', label='ADAS 1-> 2+ (Dere et al.)', color = 'green')
plt.loglog(Te_2, rate_2 *1e-1, '-o', label='ADAS 2-> 3+ (Dere et al.)', color = 'red')
plt.loglog(Te_3, rate_3 *1e-1, '-o', label='ADAS 3-> 4+ (Dere et al.)', color = 'cyan')


plt.legend(fontsize=15, loc= 'lower right')
plt.ylim(1e-15, 1e-6)
plt.xlim(0, 1e4)
plt.title('Argon electron impact ionization Rate Coefficients', fontsize=16, fontweight='bold')

e_eV = np.array([
    1.576000e+1, 1.800000e+1, 1.900000e+1, 2.000000e+1, 2.100000e+1,
    2.200000e+1, 2.300000e+1, 2.400000e+1, 2.500000e+1, 3.000000e+1,
    4.000000e+1, 5.000000e+1, 6.000000e+1, 7.000000e+1, 8.000000e+1,
    9.000000e+1, 1.000000e+2, 1.250000e+2, 1.500000e+2, 1.750000e+2,
    2.000000e+2, 2.250000e+2, 2.500000e+2, 2.750000e+2, 3.000000e+2
])

sigma_m2 = np.array([
    0.000000e+0, 1.277200e-21, 2.074700e-21, 3.219900e-21, 4.408700e-21,
    5.405200e-21, 6.672000e-21, 7.663000e-21, 8.700000e-21, 1.225100e-20,
    1.725600e-20, 1.951700e-20, 2.117900e-20, 2.226900e-20, 2.283400e-20,
    2.377700e-20, 2.457600e-20, 2.436600e-20, 2.314200e-20, 2.190200e-20,
    2.073500e-20, 1.967200e-20, 1.871200e-20, 1.769100e-20, 1.724900e-20
])

plt.xlabel('Electron temperature $T_e$ [eV]')
plt.ylabel('Rate coefficient $k(T_e)$ [m³/s]')
plt.legend()
plt.grid(True, which='both') 
plt.show()



# Ratio DI / EA

In [190]:
# H ratio #
plt.figure(figsize=(10, 6))

ratio_DI_EA_H = model(Te, 7.1e-15 * 1e6, 13.6, 0.4) / model(Te, 12.4e-15 * 1e6, 13.6, 0.3)
plt.plot(Te, ratio_DI_EA_H, 'b-', label='DI / EAU H (Broks et al. )')
plt.xlim(1, 1e4)
plt.grid(True, which='both', ls=':', alpha=0.3)
plt.xlabel("Electron Temperature (eV)", fontsize=14)
plt.ylabel("Ratio DI / EAU", fontsize=14)
plt.legend(fontsize=15, loc='upper right')
plt.title('Electron impact ionization DI to EAU Ratio', fontsize=16, fontweight='bold')

def ratio(Te, a, b, c):
    return a * np.exp(-b / Te) / (1 + c / Te)


# Nitrogen #

In [159]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sc

colors = ['b', 'g', 'r', 'k']
plt.figure(figsize=(10, 6))

a_N0, b_N0, c_N0, P_N0 = 3.4, 0.83, 0.22, 14.53
a_N1, b_N1, c_N1, P_N1 = 4.1, 0.46, 0.62, 29.6
a_N2, b_N2, c_N2, P_N2 = 3.8, 0.6, 0.4, 47.445
a_N3, b_N3, c_N3, P_N3 = 4.0, 0.75, 0.5, 77.47

coeff_N = {'N0': (a_N0, b_N0, c_N0, P_N0),
            'N1': (a_N1, b_N1, c_N1, P_N1),
            'N2': (a_N2, b_N2, c_N2, P_N2),
            'N3': (a_N3, b_N3, c_N3, P_N3)}

def rate_coefficient(T, a, b, c, P):
    """
    Compute the rate coefficient S (in cm^3 s^-1).
    """
    a = np.asarray(a)
    b = np.asarray(b)
    c = np.asarray(c)
    P = np.asarray(P)

    def exp_func(x):
        return np.exp(-x) / x
    
    int_1 = sc.integrate.quad(exp_func, P / T, np.inf)[0]
    int_2 = sc.integrate.quad(exp_func, P / T + c, np.inf)[0]
    
    S = 6.7e-7 * a / T**1.5 * 1/(P / T) * (int_1 - b * np.exp(c) / (P/T + c) * int_2)

    return S

Te_plot = np.logspace(0, 4, 1000)

for i, (state, coeffs) in enumerate(coeff_N.items()):
    a, b, c, P = coeffs
    S_plot = [rate_coefficient(T, a, b, c, P) for T in Te_plot]
    plt.loglog(Te_plot, S_plot, label=f'Lotz {state} -> {state[:-1]}+', lw=2, color = colors[i])
    plt.fill_between(Te_plot, np.array(S_plot)*1.4, np.array(S_plot)*0.7, alpha=0.2, color=colors[i])

# Bell Data # 

x_1 = [1.2299505200127774, 1.273118748676529, 1.295267744355328, 1.317802075659802, 1.3407284464399383, 1.3640536771754934, 1.4119285957903747, 1.4614838059295072, 1.5931152357256129, 1.766814729614692, 1.9594529126209963, 2.2109009606367502, 2.582171115409593, 3.1759427061600025, 4.332156326836239, 6.012100298738503, 8.786587226230273, 13.523407734222115, 23.893355979408877, 50.16196396224763, 94.95724149488028, 217.3094695455825, 448.4204325419173, 1099.508274046798, 2117.5939944708894, 5374.487766218062, 9495.724149488027]
y_1 = [1.0226811702720644e-13, 1.223664434788464e-13, 1.53131640471704e-13, 1.8738174228603867e-13, 2.3449299313555304e-13, 3.069113764883905e-13, 3.8407459778154424e-13, 6.290733667073061e-13, 1.5777999866581129e-12, 3.3823673953184594e-12, 1.0857111194022017e-11, 2.848035868435805e-11, 7.813707376518085e-11, 2.0496906107066367e-10, 8.420243975795665e-10, 2.5842686689795357e-9, 6.932806692144509e-9, 1.5896397032896162e-8, 3.485040201206771e-8, 6.24387996565362e-8, 8.357529574264813e-8, 1e-7, 1e-7, 9.34930269415822e-8, 7.990931403980979e-8, 6.38549847031294e-8, 5.581529786570688e-8]

x_2 = [2.6727989507129006, 3.01578758522739, 3.3446029561960655, 3.8394557167789056, 4.407524717933084, 5.237223688824735, 6.33137631642622, 7.6541176092477485, 9.744600632908476, 13.064863267737023, 20.45791751463052, 34.91975283321503, 67.25357201348045, 167.77185295673397, 505.96427129751333, 1375.8680288160465, 4221.5133284346875, 9016.877712317013]
y_2 = [9.77821855988577e-14, 3.138724856769573e-13, 9.419459301799856e-13, 2.826823553740301e-12, 8.295282272757476e-12, 2.9786945720946926e-11, 9.561355820489454e-11, 2.1923459724834422e-10, 6.290733667073061e-10, 1.7258895017047566e-9, 4.95228003713452e-9, 1.187614044052095e-8, 2.1760172691812416e-8, 3.186007202393702e-8, 3.4077484777388155e-8, 2.9126326549087384e-8, 2.127757244814001e-8, 1.700276218763224e-8]

x_3 = [4.332156326836239, 4.722340555418246, 5.237223688824735, 6.1166953992942785, 6.901624247827922, 7.6541176092477485, 9.094974759211652, 11.38098301710062, 14.741420201106218, 20.108088432108044, 30.94829330452783, 55.63119026636751, 100, 221.09009606367502, 488.80830577445045, 974.4600632908481, 2430.903899318121, 6610.354022862211, 9333.347873177096]
y_3 = [1.0226811702720644e-13, 2.0961799924531258e-13, 6.014794296281798e-13, 2.3625261451612974e-12, 6.4816908284944795e-12, 1.486202276070645e-11, 4.264519748024758e-11, 1.2514185761897705e-10, 3.590829671796726e-10, 9.419459301799857e-10, 2.6428829066894112e-9, 5.794113696225886e-9, 8.872621353826384e-9, 1.2702701826031998e-8, 1.2990813969063467e-8, 1.2145505204027332e-8, 9.70538989520764e-9, 7.0900508611923466e-9, 6.337938955862635e-9]


x_4 = [7.922758252752512, 8.488656332548747, 9.744600632908476, 11.578982884692152, 14.741420201106218, 18.767578440147254, 24.309038993181222, 33.73571255672632, 52.82584368771092, 101.73974310737503, 262.7094259410561, 580.8245221814101, 1692.2495975112506, 5192.252513549118, 9333.347873177096]
y_4 = [1.0226811702720644e-13, 1.9597821250882564e-13, 6.151216869867208e-13, 3.023566191205483e-12, 1.519911082952933e-11, 5.1026080256496174e-11, 1.3385154135310025e-10, 3.590829671796726e-10, 1.0075039401265159e-9, 2.310129700083158e-9, 3.869566705674126e-9, 4.527350892858468e-9, 4.328761281083061e-9, 3.6998304144069656e-9, 3.1622776601683795e-9]



plt.loglog(x_1, y_1, color = colors[0], linestyle='--', label='Bell et al. 0->1+')
plt.loglog(x_2, y_2, color = colors[1], linestyle='--', label='Bell et al. 1->2+')
plt.loglog(x_3, y_3, color = colors[2], linestyle='--', label='Bell et al. 2->3+')
plt.loglog(x_4, y_4, color = colors[3], linestyle='--', label='Bell et al. 3->4+')

plt.xlabel("Electron Temperature (eV)", fontsize = 14)
plt.ylabel("Rate Coefficient (cm³/s)", fontsize = 14)
plt.title('Nitrogen electron impact ionization Rate Coefficients', fontsize=16, fontweight='bold')
plt.legend(fontsize=12, loc= 'lower right')
plt.grid(True, which='both', ls=':', alpha=0.3)
plt.ylim(1e-15, 1e-7)

In [60]:
x = [15.602836879432624, 16.31205673758865, 16.666666666666668, 18.085106382978726, 18.43971631205674, 21.631205673758867, 24.822695035460995, 30.49645390070922, 36.17021276595745, 40.42553191489362, 45.39007092198582, 50.70921985815603, 56.38297872340426, 64.8936170212766, 76.59574468085107, 95.0354609929078, 100.35460992907802]
y = [1.282051282051282e-18, 2.6923076923076924e-17, 4.7435897435897433e-17, 7.051282051282051e-17, 9.935897435897435e-17, 1.4166666666666667e-16, 1.7371794871794872e-16, 2.2179487179487179e-16, 2.7243589743589745e-16, 3.0256410256410257e-16, 3.358974358974359e-16, 3.615384615384615e-16, 3.7371794871794874e-16, 3.8076923076923076e-16, 3.75e-16, 3.5384615384615383e-16, 3.4615384615384615e-16]


# LXCat excitation cross-section data for Argon
x_ex_lxcat = [1.155000e+1, 2.000000e+1, 3.000000e+1, 4.000000e+1, 5.000000e+1, 6.000000e+1,
              7.000000e+1, 8.000000e+1, 9.000000e+1, 1.000000e+2, 1.500000e+2, 2.000000e+2,
              3.000000e+2, 5.000000e+2, 7.000000e+2, 1.000000e+3, 2.000000e+3, 3.000000e+3, 4.000000e+3]

y_ex_lxcat = [0.000000e+0, 5.090000e-21, 7.450000e-21, 7.880000e-21, 7.700000e-21, 7.560000e-21,
              7.370000e-21, 7.190000e-21, 7.080000e-21, 7.010000e-21, 6.470000e-21, 5.380000e-21,
              4.080000e-21, 2.830000e-21, 2.190000e-21, 1.670000e-21, 9.630000e-22, 6.920000e-22, 5.460000e-22]

# LXCat ionization cross-section data for Argon
x_ion_lxcat = [1.575900e+1, 1.600000e+1, 1.700000e+1, 1.800000e+1, 1.900000e+1, 2.000000e+1,
               2.100000e+1, 2.200000e+1, 2.300000e+1, 2.400000e+1, 2.500000e+1, 2.600000e+1,
               2.800000e+1, 3.000000e+1, 3.200000e+1, 3.400000e+1, 3.600000e+1, 3.800000e+1,
               4.000000e+1, 4.250000e+1, 4.500000e+1, 5.000000e+1, 5.500000e+1, 6.000000e+1,
               6.500000e+1, 7.000000e+1, 7.500000e+1, 8.000000e+1, 8.500000e+1, 9.000000e+1,
               1.000000e+2, 1.150000e+2, 1.300000e+2, 1.450000e+2, 1.600000e+2, 1.800000e+2,
               2.000000e+2, 4.000000e+2, 7.000000e+2, 1.000000e+3]

y_ion_lxcat = [0.000000e+0, 2.023000e-22, 1.337000e-21, 2.938000e-21, 4.601000e-21, 6.272000e-21,
               7.873000e-21, 9.325000e-21, 1.056000e-20, 1.179000e-20, 1.302000e-20, 1.408000e-20,
               1.601000e-20, 1.803000e-20, 1.962000e-20, 2.111000e-20, 2.243000e-20, 2.331000e-20,
               2.393000e-20, 2.446000e-20, 2.490000e-20, 2.534000e-20, 2.595000e-20, 2.657000e-20,
               2.727000e-20, 2.771000e-20, 2.815000e-20, 2.841000e-20, 2.850000e-20, 2.859000e-20,
               2.850000e-20, 2.824000e-20, 2.762000e-20, 2.709000e-20, 2.622000e-20, 2.516000e-20,
               2.393000e-20, 1.690000e-20, 1.170000e-20, 9.270000e-21]

plt.figure(figsize=(10, 6))
plt.plot(x_ion_lxcat, np.array(y_ion_lxcat) * 1e4, 'o-', label='LXCat Ionization Data', markersize=5)

plt.plot(x_ex_lxcat, np.array(y_ex_lxcat) * 1e4, 's-', label='LXCat Excitation Data', markersize=5)

from scipy.interpolate import interp1d

f_ex_interp= interp1d(x_ex_lxcat, y_ex_lxcat, bounds_error=False, fill_value=0.0)

y_ex_interp = f_ex_interp(x_ion_lxcat)
total_lxcat = np.array(y_ion_lxcat) * 1e4 + np.array(y_ex_interp) * 1e4

plt.plot(x_ion_lxcat, total_lxcat, 'b^-', label='LXCat Total (Excitation + Ionization)', markersize=6)

plt.plot(x, y, 'ro-', label='Experimental Data', markersize=6)
plt.xlabel('Electron Energy (eV)')
plt.ylabel('Cross Section (cm²)')
plt.title('Electron Impact Ionization Cross Section for Argon')
plt.grid(True)
plt.legend()
plt.xlim(10, 100)
plt.yscale('log')

## Argon ##

In [161]:
import numpy as np
import matplotlib.pyplot as plt

# Electron temperatures in eV for ISEL=1 and ISEL=2
Te1 = np.array([0.1723, 0.4308, 0.8617, 1.723, 4.308, 8.617, 17.23, 43.08, 86.17, 172.3, 430.8, 861.7])
Te2 = np.array([0.6894, 1.723, 3.447, 6.894, 17.23, 34.47, 68.94, 172.3, 344.7, 689.4, 1723., 3447.])

# Corresponding ionization rate coefficients in m^3/s
rate1 = np.array([1e-36, 1.23e-23, 1.782e-15, 2.426e-11, 7.985e-09, 5.44e-08, 1.382e-07, 2.355e-07, 2.803e-07, 3.247e-07, 5.391e-07, 8.127e-07])
rate2 = np.array([7.161e-26, 3.458e-15, 1.498e-11, 1.116e-09, 1.527e-08, 3.643e-08, 5.546e-08, 6.736e-08, 6.665e-08, 6.035e-08, 4.83e-08, 3.889e-08])


# Electron temperatures in eV for the first ISEL level (Ar+0 -> Ar+1)
Te_bis = np.array([0.8626, 1.171, 2.523, 4.101, 5.964, 8.193, 10.90, 14.26, 18.52, 24.06, 31.52, 42.02, 57.63, 82.67, 127.4, 220.7, 472.8, 1570., 16110., 86520.])

# Corresponding ionization rate coefficients in m^3/s
rate_bis = np.array([1.340e-16, 2.079e-14, 5.285e-11, 8.648e-10, 3.790e-09, 9.585e-09, 1.830e-08, 2.964e-08, 4.323e-08, 5.876e-08, 7.579e-08, 9.381e-08, 1.124e-07, 1.309e-07, 1.477e-07, 1.582e-07, 1.553e-07, 1.290e-07, 6.628e-08, 3.646e-08])

# Plot
plt.figure(figsize=(12,8))
plt.loglog(Te1, rate1 * 1e-7, 'o-', label='ADAS ca09')
plt.loglog(Te_bis, rate_bis * 1e-7, 's-', label='ADAS dere09')
labels = ['Exciation ionization LxCat', 'Direct ionization LxCat']
for process, k in rate_coeffs.items():
    plt.loglog(Te_eV, k, label=labels.pop(0))
plt.loglog(Te_eV, total_rate, 'k--', label='TOTAL (LXCat)')
plt.xlabel('Electron Temperature $T_e$ [eV]')
plt.ylabel('Ionization rate coefficient [m³/s]')
plt.title('Comparison of Argon Ionization Rates from different Sources', fontsize = 16, fontweight='bold')
plt.grid(True, which='both', ls='--')

plt.xlim(1, 2*1e3)
plt.ylim(1e-19, 1e-12)


# Constants
A = 0.585e-14  # cm^2 eV^2
I = 15.76      # eV
m_e = 9.10938356e-28  # g
k_B = 1.0      # eV (we keep T_e in eV)

# Prefactor for rate coefficient
prefactor = 2 * np.sqrt(2) * A / np.sqrt(np.pi * m_e)  # in cm^3/s * eV^-0.5

# Electron temperature array (eV)
Te = np.logspace(0, 3, 200)  # 1 to 1e3 eV

# Integral approximation using expi function
# ∫_x0^∞ e^{-x} ln(x) dx = -expint(-x0), but we can approximate numerically with expi
def rate_coefficient(T):
    x0 = I / T
    # Integral: ∫_{x0}^{∞} ln(x) e^{-x} dx = -gamma - ln(x0) - Ei(-x0)?
    # We'll compute numerically for safety
    from scipy.integrate import quad
    integrals = []
    for x in x0 if np.iterable(x0) else [x0]:
        integral, _ = quad(lambda t: np.log(t) * np.exp(-t), x, np.inf)
        integrals.append(integral)
    integrals = np.array(integrals)
    return prefactor / np.sqrt(T) * (np.exp(-I/T) * np.log(T/I) + integrals)

rate = rate_coefficient(Te)
plt.loglog(Te, rate * 1e-12, lw=2, label ='Lotz Formula')
plt.legend(fontsize=14, loc = 'lower right')


plt.figure(figsize=(12,8))
